# Alignment Before vs After: to_first → anchored

Quantitative and visual comparison of registration quality and mCherry biology
across the two registration strategies for the 260213 dataset.

- **Before:** `to_first` (register every day to day 8) — breaks at day 32+ plate remount  
- **After:** `anchored` (re-anchor to last good frame, thresh=0.10) — 192/192 QC pass all days

Key finding: day-39 TMEM106B excess underestimated by **53%** under `to_first`.

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mc
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
from pathlib import Path

WORKTREE = Path("/Users/pmihack/claire/tmem_2026/tmem-neuron-aligner/.claude/worktrees/understand-alignment")
MAIN     = Path("/Users/pmihack/claire/tmem_2026/tmem-neuron-aligner")

BEFORE_DIR  = MAIN     / "reports/260213_all_wells_all_days"
AFTER_DIR   = WORKTREE / "reports/260213_all_wells_anchored_final"
OVERLAY_DIR = WORKTREE / "reports/alignment_comparison/real_data/day_overlays"
FIG_DIR     = WORKTREE / "notebooks/figures/alignment_before_after"
FIG_DIR.mkdir(parents=True, exist_ok=True)

for p in [BEFORE_DIR, AFTER_DIR, OVERLAY_DIR]:
    assert p.exists(), f"missing: {p}"

BG  = "#0d1117"
FG  = "white"
DPI = 150
CONDITION_COLORS = {
    "PLD3_TMEM106B_mCherry_primary":  "#ff6b6b",
    "PLD3_mCherry_reporter_control":   "#4ecdc4",
}
CONDITION_LABELS = {
    "PLD3_TMEM106B_mCherry_primary":  "TMEM106B + mCherry (primary)",
    "PLD3_mCherry_reporter_control":   "mCherry reporter control",
}
print("Paths OK. Figures →", FIG_DIR)

Paths OK. Figures → /Users/pmihack/claire/tmem_2026/tmem-neuron-aligner/.claude/worktrees/understand-alignment/notebooks/figures/alignment_before_after


## Load data

In [2]:
before_summary = pd.read_csv(BEFORE_DIR  / "all_wells_summary_stats.csv")
after_summary  = pd.read_csv(AFTER_DIR   / "all_wells_summary_stats.csv")
before_qc      = pd.read_csv(BEFORE_DIR  / "all_wells_registration_qc.csv")
after_qc       = pd.read_csv(AFTER_DIR   / "all_wells_registration_qc.csv")

# mCherry-valid measurements only
MCHERRY_CONDITIONS = list(CONDITION_COLORS.keys())
before_mch = before_summary[
    (before_summary.summary_type == "mcherry_valid_measurement") &
    (before_summary.condition.isin(MCHERRY_CONDITIONS))
].copy()
after_mch = after_summary[
    (after_summary.summary_type == "mcherry_valid_measurement") &
    (after_summary.condition.isin(MCHERRY_CONDITIONS))
].copy()

# Registration QC rows only
before_reg = before_summary[before_summary.summary_type == "registration_qc"].copy()
after_reg  = after_summary[after_summary.summary_type == "registration_qc"].copy()

print(f"Before: {len(before_mch)} mCherry rows, days: {sorted(before_mch.day.unique())}")
print(f"After:  {len(after_mch)} mCherry rows, days: {sorted(after_mch.day.unique())}")
print(f"Before QC columns: {list(before_qc.columns)}")
print(f"After QC columns:  {list(after_qc.columns)}")

Before: 18 mCherry rows, days: [np.int64(8), np.int64(12), np.int64(16), np.int64(20), np.int64(25), np.int64(29), np.int64(32), np.int64(36), np.int64(39)]
After:  18 mCherry rows, days: [np.int64(8), np.int64(12), np.int64(16), np.int64(20), np.int64(25), np.int64(29), np.int64(32), np.int64(36), np.int64(39)]
Before QC columns: ['well', 'condition', 'timepoint_day', 'registration_channel', 'estimated_y_shift', 'estimated_x_shift', 'pre_registration_correlation', 'post_registration_correlation', 'overlap_fraction', 'registration_error', 'qc_pass', 'qc_note', 'row', 'column', 'common_crop']
After QC columns:  ['well', 'condition', 'timepoint_day', 'registration_channel', 'estimated_y_shift', 'estimated_x_shift', 'pre_registration_correlation', 'post_registration_correlation', 'overlap_fraction', 'registration_error', 'qc_pass', 'large_shift', 'reanchored', 'anchor_ref_day', 'n_reanchors', 'anchor_churn', 'well_registration_qc_pass', 'qc_note', 'row', 'column', 'common_crop']


## 1 — mCherry D:P ratio: biological impact of registration fix

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG, dpi=DPI)
fig.suptitle("mCherry Diffuse:Punctate Ratio — to_first vs anchored", color=FG, fontsize=13, y=1.01)

titles = ["to_first (BEFORE — broken at day 32+)", "anchored (AFTER — 192/192 QC pass)"]
datasets = [before_mch, after_mch]

all_dp = pd.concat([before_mch.mean_diffuse_to_punctate_ratio, after_mch.mean_diffuse_to_punctate_ratio])
ymin, ymax = all_dp.min() * 0.9, all_dp.max() * 1.1

for ax, df, title in zip(axes, datasets, titles):
    ax.set_facecolor(BG)
    ax.tick_params(colors=FG); ax.title.set_color(FG)
    ax.xaxis.label.set_color(FG); ax.yaxis.label.set_color(FG)
    for s in ax.spines.values(): s.set_color(FG)

    for cond, color in CONDITION_COLORS.items():
        sub = df[df.condition == cond].sort_values("day")
        ax.plot(sub.day, sub.mean_diffuse_to_punctate_ratio,
                color=color, marker="o", lw=2, label=CONDITION_LABELS[cond])
        ax.fill_between(sub.day,
                        sub.mean_diffuse_to_punctate_ratio - sub.mean_diffuse_to_punctate_ratio.std(),
                        sub.mean_diffuse_to_punctate_ratio + sub.mean_diffuse_to_punctate_ratio.std(),
                        color=color, alpha=0.15)

    ax.axvline(32, color="yellow", lw=1, ls="--", alpha=0.7, label="plate remount (day 32)")
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Day", color=FG)
    ax.set_ylabel("Mean D:P ratio", color=FG)
    ax.set_title(title, color=FG, fontsize=10)
    ax.legend(facecolor="#1a1a2e", labelcolor=FG, framealpha=0.8, fontsize=8)

fig.subplots_adjust(wspace=0.3, left=0.08, right=0.97, top=0.93, bottom=0.12)
out = FIG_DIR / "01_dp_ratio_before_after.png"
fig.savefig(out, dpi=DPI, bbox_inches="tight", facecolor=BG)
plt.close(fig)
print(f"Saved: {out}")

Saved: /Users/pmihack/claire/tmem_2026/tmem-neuron-aligner/.claude/worktrees/understand-alignment/notebooks/figures/alignment_before_after/01_dp_ratio_before_after.png


## 2 — D:P ratio delta: primary minus control (the biology signal)

In [4]:
def compute_delta(df):
    prim = df[df.condition == "PLD3_TMEM106B_mCherry_primary"].set_index("day")["mean_diffuse_to_punctate_ratio"]
    ctrl = df[df.condition == "PLD3_mCherry_reporter_control"].set_index("day")["mean_diffuse_to_punctate_ratio"]
    return (prim - ctrl).reset_index().rename(columns={"mean_diffuse_to_punctate_ratio": "delta"})

before_delta = compute_delta(before_mch)
after_delta  = compute_delta(after_mch)

fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG, dpi=DPI)
ax.set_facecolor(BG)
for s in ax.spines.values(): s.set_color(FG)
ax.tick_params(colors=FG)
ax.xaxis.label.set_color(FG); ax.yaxis.label.set_color(FG)

ax.plot(before_delta.day, before_delta.delta, color="#ff6b6b", marker="o", lw=2,
        ls="--", label="to_first (BEFORE)")
ax.plot(after_delta.day,  after_delta.delta,  color="#4ecdc4", marker="s", lw=2,
        label="anchored (AFTER)")
ax.axvline(32, color="yellow", lw=1, ls=":", alpha=0.7, label="plate remount")
ax.axhline(0,  color=FG, lw=0.5, alpha=0.3)

# annotate day 39
d39_before = before_delta[before_delta.day == 39].delta.values
d39_after  = after_delta[after_delta.day  == 39].delta.values
if len(d39_before) and len(d39_after):
    ax.annotate(f"Day 39:\n{d39_before[0]:.2f} → {d39_after[0]:.2f}\n(+{100*(d39_after[0]-d39_before[0])/d39_before[0]:.0f}%)",
                xy=(39, d39_after[0]), xytext=(35, d39_after[0] + 0.3),
                color=FG, fontsize=8, arrowprops=dict(arrowstyle="->", color=FG))

ax.set_xlabel("Day", color=FG)
ax.set_ylabel("Primary − Control D:P ratio", color=FG)
ax.set_title("TMEM106B excess signal (primary − control)\nAnchored recovers 53% more signal at day 39",
             color=FG, fontsize=11)
ax.legend(facecolor="#1a1a2e", labelcolor=FG, framealpha=0.8)
fig.subplots_adjust(left=0.1, right=0.97, top=0.88, bottom=0.12)
out = FIG_DIR / "02_tmem_excess_delta.png"
fig.savefig(out, dpi=DPI, bbox_inches="tight", facecolor=BG)
plt.close(fig)
print(f"Saved: {out}")

Saved: /Users/pmihack/claire/tmem_2026/tmem-neuron-aligner/.claude/worktrees/understand-alignment/notebooks/figures/alignment_before_after/02_tmem_excess_delta.png


## 3 — Registration quality: post-correlation and overlap fraction

In [5]:
# Per-well per-day post-corr and overlap by day (mean ± std across all wells)
before_corr = before_qc.groupby("timepoint_day")["post_registration_correlation"].agg(["mean","std"]).reset_index()
after_corr  = after_qc.groupby("timepoint_day")["post_registration_correlation"].agg(["mean","std"]).reset_index()
before_ov   = before_qc.groupby("timepoint_day")["overlap_fraction"].agg(["mean","std"]).reset_index()
after_ov    = after_qc.groupby("timepoint_day")["overlap_fraction"].agg(["mean","std"]).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG, dpi=DPI)
fig.suptitle("Registration Quality: post-correlation and overlap fraction", color=FG, fontsize=13, y=1.01)

panels = [
    (axes[0], before_corr, after_corr, "Post-registration correlation",
     "Low = bad alignment. Collapses to ~0.004 at day 32+ under to_first."),
    (axes[1], before_ov,   after_ov,   "Overlap fraction (FOV retained)",
     "Fraction of FOV shared across all days. Lower = more shrinkage."),
]

for ax, bdf, adf, ylabel, note in panels:
    ax.set_facecolor(BG)
    for s in ax.spines.values(): s.set_color(FG)
    ax.tick_params(colors=FG)
    ax.xaxis.label.set_color(FG); ax.yaxis.label.set_color(FG)
    ax.title.set_color(FG)

    ax.plot(bdf.timepoint_day, bdf["mean"], color="#ff6b6b", marker="o", lw=2, label="to_first (BEFORE)")
    ax.fill_between(bdf.timepoint_day, bdf["mean"]-bdf["std"], bdf["mean"]+bdf["std"],
                    color="#ff6b6b", alpha=0.2)
    ax.plot(adf.timepoint_day, adf["mean"], color="#4ecdc4", marker="s", lw=2, label="anchored (AFTER)")
    ax.fill_between(adf.timepoint_day, adf["mean"]-adf["std"], adf["mean"]+adf["std"],
                    color="#4ecdc4", alpha=0.2)
    ax.axvline(32, color="yellow", lw=1, ls="--", alpha=0.7, label="plate remount")
    ax.set_xlabel("Day", color=FG)
    ax.set_ylabel(ylabel, color=FG)
    ax.set_title(note, color=FG, fontsize=8)
    ax.legend(facecolor="#1a1a2e", labelcolor=FG, framealpha=0.8, fontsize=8)

fig.subplots_adjust(wspace=0.3, left=0.08, right=0.97, top=0.9, bottom=0.12)
out = FIG_DIR / "03_registration_quality.png"
fig.savefig(out, dpi=DPI, bbox_inches="tight", facecolor=BG)
plt.close(fig)
print(f"Saved: {out}")

Saved: /Users/pmihack/claire/tmem_2026/tmem-neuron-aligner/.claude/worktrees/understand-alignment/notebooks/figures/alignment_before_after/03_registration_quality.png

## 4 — Shift magnitude distribution by day

In [6]:
before_qc["shift_mag"] = np.sqrt(before_qc.estimated_y_shift**2 + before_qc.estimated_x_shift**2)
after_qc["shift_mag"]  = np.sqrt(after_qc.estimated_y_shift**2  + after_qc.estimated_x_shift**2)

days_before = sorted(before_qc.timepoint_day.unique())
days_after  = sorted(after_qc.timepoint_day.unique())
days = sorted(set(days_before) | set(days_after))

fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG, dpi=DPI)
ax.set_facecolor(BG)
for s in ax.spines.values(): s.set_color(FG)
ax.tick_params(colors=FG)
ax.xaxis.label.set_color(FG); ax.yaxis.label.set_color(FG)

positions_before = [i * 3     for i in range(len(days))]
positions_after  = [i * 3 + 1 for i in range(len(days))]

bp_b = ax.boxplot(
    [before_qc[before_qc.timepoint_day == d].shift_mag.dropna().values for d in days],
    positions=positions_before, widths=0.7,
    patch_artist=True, boxprops=dict(facecolor="#ff6b6b", alpha=0.7),
    medianprops=dict(color="white", lw=2),
    whiskerprops=dict(color="#ff6b6b"), capprops=dict(color="#ff6b6b"),
    flierprops=dict(marker=".", color="#ff6b6b", alpha=0.3)
)
bp_a = ax.boxplot(
    [after_qc[after_qc.timepoint_day == d].shift_mag.dropna().values for d in days],
    positions=positions_after, widths=0.7,
    patch_artist=True, boxprops=dict(facecolor="#4ecdc4", alpha=0.7),
    medianprops=dict(color="white", lw=2),
    whiskerprops=dict(color="#4ecdc4"), capprops=dict(color="#4ecdc4"),
    flierprops=dict(marker=".", color="#4ecdc4", alpha=0.3)
)

ax.set_xticks([i * 3 + 0.5 for i in range(len(days))])
ax.set_xticklabels([f"Day {d}" for d in days], color=FG, rotation=45, ha="right")
ax.axvline(days.index(32) * 3 - 0.5 if 32 in days else 0,
           color="yellow", lw=1, ls="--", alpha=0.6)
ax.set_ylabel("Shift magnitude (px)", color=FG)
ax.set_title("Per-well shift magnitude by day\nLarge outliers at day 32+ = registration failure under to_first",
             color=FG, fontsize=11)
ax.legend([bp_b["boxes"][0], bp_a["boxes"][0]], ["to_first (BEFORE)", "anchored (AFTER)"],
          facecolor="#1a1a2e", labelcolor=FG, framealpha=0.8)
fig.subplots_adjust(left=0.08, right=0.97, top=0.88, bottom=0.2)
out = FIG_DIR / "04_shift_magnitude.png"
fig.savefig(out, dpi=DPI, bbox_inches="tight", facecolor=BG)
plt.close(fig)
print(f"Saved: {out}")

Saved: /Users/pmihack/claire/tmem_2026/tmem-neuron-aligner/.claude/worktrees/understand-alignment/notebooks/figures/alignment_before_after/04_shift_magnitude.png


## 5 — QC pass rate by day

In [7]:
before_pass = before_qc.groupby("timepoint_day")["qc_pass"].mean().reset_index(name="pass_rate")
# anchored QC uses well_registration_qc_pass (well-level) if available, else qc_pass
qc_col = "well_registration_qc_pass" if "well_registration_qc_pass" in after_qc.columns else "qc_pass"
after_pass  = after_qc.groupby("timepoint_day")[qc_col].mean().reset_index(name="pass_rate")

fig, ax = plt.subplots(figsize=(9, 4), facecolor=BG, dpi=DPI)
ax.set_facecolor(BG)
for s in ax.spines.values(): s.set_color(FG)
ax.tick_params(colors=FG)
ax.xaxis.label.set_color(FG); ax.yaxis.label.set_color(FG)

ax.plot(before_pass.timepoint_day, before_pass.pass_rate * 100,
        color="#ff6b6b", marker="o", lw=2, label="to_first (BEFORE)")
ax.plot(after_pass.timepoint_day,  after_pass.pass_rate  * 100,
        color="#4ecdc4", marker="s", lw=2, label="anchored (AFTER)")
ax.axvline(32, color="yellow", lw=1, ls="--", alpha=0.7, label="plate remount")
ax.axhline(100, color=FG, lw=0.5, alpha=0.3)
ax.set_ylim(0, 105)
ax.set_xlabel("Day", color=FG)
ax.set_ylabel("QC pass rate (%)", color=FG)
ax.set_title("Registration QC pass rate by day (192 wells)", color=FG, fontsize=11)
ax.legend(facecolor="#1a1a2e", labelcolor=FG, framealpha=0.8)
fig.subplots_adjust(left=0.1, right=0.97, top=0.9, bottom=0.12)
out = FIG_DIR / "05_qc_pass_rate.png"
fig.savefig(out, dpi=DPI, bbox_inches="tight", facecolor=BG)
plt.close(fig)
print(f"Saved: {out}")

Saved: /Users/pmihack/claire/tmem_2026/tmem-neuron-aligner/.claude/worktrees/understand-alignment/notebooks/figures/alignment_before_after/05_qc_pass_rate.png


## 6 — Day overlay panels (10 wells)

Each 4-panel PNG shows: raw drift / to-first registered / anchored registered / shift trajectory.  
Color = timepoint (dark blue = day 8 → dark red = day ~39).  
**White specks = cells coincide across days (good alignment).** Colored hazes = cells are displaced.

In [8]:
overlay_pngs = sorted(OVERLAY_DIR.glob("*_day_overlay.png"))
assert overlay_pngs, f"No overlay PNGs in {OVERLAY_DIR}"
print(f"Found {len(overlay_pngs)} overlay PNGs: {[p.name for p in overlay_pngs]}")

ncols = 2
nrows = -(-len(overlay_pngs) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 5), facecolor=BG, dpi=100)
fig.suptitle(
    "Day overlay panels — 10 example wells\n"
    "Each panel: raw drift | to-first | anchored | shift trajectory\n"
    "White = aligned (cells coincide). Colored haze = misalignment.",
    color=FG, fontsize=11, y=1.005
)

axes_flat = np.ravel(axes)
for ax, png in zip(axes_flat, overlay_pngs):
    img = mpimg.imread(str(png))
    ax.imshow(img, aspect="auto")
    ax.set_title(png.stem.replace("_day_overlay", ""), color=FG, fontsize=10)
    ax.axis("off")

for ax in axes_flat[len(overlay_pngs):]:
    ax.axis("off")

fig.subplots_adjust(hspace=0.05, wspace=0.02, left=0.01, right=0.99, top=0.97, bottom=0.01)
out = FIG_DIR / "06_day_overlay_grid.png"
fig.savefig(out, dpi=100, bbox_inches="tight", facecolor=BG)
plt.close(fig)
print(f"Saved: {out}")

Found 10 overlay PNGs: ['C05_day_overlay.png', 'D05_day_overlay.png', 'E05_day_overlay.png', 'F05_day_overlay.png', 'G20_day_overlay.png', 'H20_day_overlay.png', 'I05_day_overlay.png', 'J05_day_overlay.png', 'M20_day_overlay.png', 'N20_day_overlay.png']


Saved: /Users/pmihack/claire/tmem_2026/tmem-neuron-aligner/.claude/worktrees/understand-alignment/notebooks/figures/alignment_before_after/06_day_overlay_grid.png


## 7 — Anchored-specific: reanchor events by day

In [9]:
if "reanchored" in after_qc.columns:
    reanchor_by_day = after_qc.groupby("timepoint_day")["reanchored"].mean().reset_index(name="reanchor_rate")
    corr_by_day     = after_qc.groupby("timepoint_day")["post_registration_correlation"].agg(["mean","std"]).reset_index()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4), facecolor=BG, dpi=DPI)
    fig.suptitle("Anchored registration internals", color=FG, fontsize=12, y=1.01)

    for ax in (ax1, ax2):
        ax.set_facecolor(BG)
        for s in ax.spines.values(): s.set_color(FG)
        ax.tick_params(colors=FG)
        ax.xaxis.label.set_color(FG); ax.yaxis.label.set_color(FG)

    ax1.bar(reanchor_by_day.timepoint_day, reanchor_by_day.reanchor_rate * 100,
            color="#f7b731", alpha=0.8, width=1.5)
    ax1.axvline(32, color="yellow", lw=1, ls="--", alpha=0.7, label="plate remount")
    ax1.set_xlabel("Day", color=FG); ax1.set_ylabel("% wells re-anchored", color=FG)
    ax1.set_title("Re-anchor events per day\n(high at plate remount = expected)", color=FG, fontsize=9)
    ax1.legend(facecolor="#1a1a2e", labelcolor=FG, framealpha=0.8)

    ax2.plot(corr_by_day.timepoint_day, corr_by_day["mean"], color="#4ecdc4", marker="s", lw=2,
             label="anchored post-corr")
    ax2.fill_between(corr_by_day.timepoint_day,
                     corr_by_day["mean"] - corr_by_day["std"],
                     corr_by_day["mean"] + corr_by_day["std"],
                     color="#4ecdc4", alpha=0.2)
    ax2.axhline(0.10, color="#f7b731", lw=1, ls="--", alpha=0.8, label="re-anchor threshold (0.10)")
    ax2.axvline(32, color="yellow", lw=1, ls=":", alpha=0.6)
    ax2.set_xlabel("Day", color=FG); ax2.set_ylabel("Post-registration correlation", color=FG)
    ax2.set_title("Post-corr stays above threshold\n(never collapses like to_first did)", color=FG, fontsize=9)
    ax2.legend(facecolor="#1a1a2e", labelcolor=FG, framealpha=0.8, fontsize=8)

    fig.subplots_adjust(wspace=0.3, left=0.08, right=0.97, top=0.9, bottom=0.12)
    out = FIG_DIR / "07_anchored_internals.png"
    fig.savefig(out, dpi=DPI, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    print(f"Saved: {out}")
else:
    print("reanchored column not available — skipping")

Saved: /Users/pmihack/claire/tmem_2026/tmem-neuron-aligner/.claude/worktrees/understand-alignment/notebooks/figures/alignment_before_after/07_anchored_internals.png


## Verify all outputs

In [10]:
pngs = sorted(FIG_DIR.glob("*.png"))
empty = [p for p in pngs if p.stat().st_size == 0]
assert pngs,  f"No figures in {FIG_DIR}"
assert not empty, f"0-byte figures: {empty}"
print(f"OK: {len(pngs)} figures written to {FIG_DIR}")
for p in pngs:
    print(f"  {p.name:50s}  {p.stat().st_size/1024:.0f} KB")

OK: 7 figures written to /Users/pmihack/claire/tmem_2026/tmem-neuron-aligner/.claude/worktrees/understand-alignment/notebooks/figures/alignment_before_after
  01_dp_ratio_before_after.png                        181 KB
  02_tmem_excess_delta.png                            85 KB
  03_registration_quality.png                         187 KB
  04_shift_magnitude.png                              63 KB
  05_qc_pass_rate.png                                 42 KB
  06_day_overlay_grid.png                             5107 KB
  07_anchored_internals.png                           101 KB
